# Feature Engineering

## Overview

Feature engineering transforms raw variables into representations that better expose the underlying structure to a model. Good features often matter more than model choice.

**Categories of feature engineering:**

| Category | Examples |
|---|---|
| Numeric transforms | Log, sqrt, Box-Cox, binning, polynomial |
| Interactions | Product terms, ratio features |
| Temporal | Lag features, rolling statistics, cyclic encoding |
| Categorical | Target encoding, frequency encoding, ordinal mapping |
| Domain-specific | Nutrient ratios, diversity indices, load calculations |

**Key principle:** features must be engineered using only training-set statistics. Any feature that uses information from the validation or test set (e.g. target encoding fitted on all data) constitutes leakage. Always engineer features inside a pipeline or fit encoders on training data only.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, PowerTransformer,
    KBinsDiscretizer, PolynomialFeatures, TargetEncoder)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)
n = 500
df = pd.DataFrame({
    'elevation':   rng.uniform(50, 400, n),
    'nitrate':     rng.gamma(2, 2, n),          # right-skewed
    'phosphorus':  rng.gamma(1.5, 1.5, n),      # right-skewed
    'flow_rate':   rng.lognormal(1.5, 0.8, n),  # log-normal
    'catchment':   rng.choice(['North','South','East','West','Central'], n),
    'month':       rng.integers(1, 13, n),
})
df['richness'] = (25 - 0.04*df['elevation'] - 0.8*df['nitrate']
                  + 2.0*(df['nitrate']/df['phosphorus'].clip(0.1))
                  + rng.normal(0, 3, n)).clip(0)
X = df.drop('richness', axis=1)
y = df['richness']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train={len(X_tr)}, Test={len(X_te)}")

---
## Numeric Transforms: Skewness Correction

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14,7))
for ax, col in zip(axes[0], ['nitrate','phosphorus','flow_rate']):
    ax.hist(X_tr[col], bins=30, color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_title(f'{col} (raw)  skew={X_tr[col].skew():.2f}')
# Log transform
log_cols = ['nitrate','phosphorus','flow_rate']
X_log = X_tr.copy()
for col in log_cols:
    X_log[f'log_{col}'] = np.log1p(X_tr[col])
for ax, col in zip(axes[1], log_cols):
    ax.hist(X_log[f'log_{col}'], bins=30, color='#e74c3c', alpha=0.7, edgecolor='white')
    ax.set_title(f'log1p({col})  skew={X_log[f"log_{col}"].skew():.2f}')
plt.suptitle('Log Transform Corrects Right Skew')
plt.tight_layout(); plt.show()

# Box-Cox via PowerTransformer
pt = PowerTransformer(method='box-cox', standardize=True)
X_bc = pt.fit_transform(X_tr[['nitrate','phosphorus']].clip(lower=0.01))
print("Box-Cox lambdas:", pt.lambdas_.round(3))
print("(lambda~1: no transform needed; lambda~0: log transform optimal)")

---
## Interaction and Ratio Features

In [ ]:
# Domain-informed ratio features
X_fe = X_tr.copy()
X_fe['N_P_ratio']     = X_tr['nitrate'] / X_tr['phosphorus'].clip(0.01)
X_fe['nutrient_load'] = X_tr['nitrate'] * np.log1p(X_tr['flow_rate'])
X_fe['elev_nitrate']  = X_tr['elevation'] * X_tr['nitrate']  # interaction
# Cyclic encoding for month (preserves Jan-Dec continuity)
X_fe['month_sin'] = np.sin(2*np.pi*X_tr['month']/12)
X_fe['month_cos'] = np.cos(2*np.pi*X_tr['month']/12)

# Demonstrate cyclic encoding preserves circularity
fig, axes = plt.subplots(1, 2, figsize=(11,4))
theta = np.linspace(0, 2*np.pi, 100)
axes[0].scatter(X_fe['month_sin'], X_fe['month_cos'],
               c=X_tr['month'], cmap='hsv', s=20, alpha=0.6)
axes[0].set_xlabel('sin(month)'); axes[0].set_ylabel('cos(month)')
axes[0].set_title('Cyclic Month Encoding: Circle Preserves Jan-Dec Adjacency')
# Correlation of new features with target
new_feats = ['N_P_ratio','nutrient_load','elev_nitrate','month_sin','month_cos']
corrs = [X_fe[f].corr(y_tr) for f in new_feats]
axes[1].barh(new_feats, corrs, color=['steelblue' if c>0 else '#e74c3c' for c in corrs])
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('Correlation of Engineered Features with Richness')
plt.tight_layout(); plt.show()

---
## Target Encoding for Categorical Variables

In [ ]:
# Target encoding: replace category with mean of target within that category
# MUST be fit on training data only -- leakage risk if fit on full data
from sklearn.preprocessing import TargetEncoder
te = TargetEncoder(smooth='auto', target_type='continuous', random_state=42)
catchment_encoded_tr = te.fit_transform(
    X_tr[['catchment']], y_tr).ravel()
catchment_encoded_te = te.transform(X_te[['catchment']]).ravel()
print("Target encoding (catchment mean richness):")
for cat in X_tr['catchment'].unique():
    mask = X_tr['catchment'] == cat
    raw_mean = y_tr[mask].mean()
    encoded_val = catchment_encoded_tr[mask].mean()
    print(f"  {cat:8s}: raw mean={raw_mean:.2f}, smoothed encoded={encoded_val:.2f}")
print("\nSmoothing shrinks low-n categories toward the global mean")
print("High-n categories stay close to their observed mean")

In [ ]:
# Full feature engineering pipeline
from sklearn.preprocessing import FunctionTransformer

def add_ratio_features(X):
    X = X.copy()
    X['N_P_ratio']     = X['nitrate'] / X['phosphorus'].clip(0.01)
    X['nutrient_load'] = X['nitrate'] * np.log1p(X['flow_rate'])
    X['month_sin']     = np.sin(2*np.pi*X['month']/12)
    X['month_cos']     = np.cos(2*np.pi*X['month']/12)
    return X

# Compare: baseline vs with feature engineering
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

baseline_pipe = Pipeline([
    ('prep', ColumnTransformer([
        ('num', StandardScaler(), ['elevation','nitrate','phosphorus','flow_rate','month']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['catchment']),
    ])),
    ('reg', GradientBoostingRegressor(n_estimators=200, random_state=42)),
])
baseline_r2 = cross_val_score(baseline_pipe, X_tr, y_tr, cv=5, scoring='r2').mean()

# With engineered features (domain-informed)
X_tr_fe = add_ratio_features(X_tr)
X_te_fe = add_ratio_features(X_te)
fe_pipe = Pipeline([
    ('prep', ColumnTransformer([
        ('num', StandardScaler(),
         ['elevation','nitrate','phosphorus','flow_rate','N_P_ratio','nutrient_load']),
        ('cyc', 'passthrough', ['month_sin','month_cos']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['catchment']),
    ])),
    ('reg', GradientBoostingRegressor(n_estimators=200, random_state=42)),
])
fe_r2 = cross_val_score(fe_pipe, X_tr_fe, y_tr, cv=5, scoring='r2').mean()
print(f"Baseline CV R2:           {baseline_r2:.3f}")
print(f"With feature engineering: {fe_r2:.3f}")
print(f"Improvement: {fe_r2-baseline_r2:+.3f}")

---

## Common Pitfalls

**1. Target encoding fitted on the full dataset before CV**  
Fitting a target encoder on all n observations before cross-validation encodes the target mean into the feature — the model can effectively look up the outcome. Always fit target encoders inside a pipeline so they see only the training fold during CV.

**2. Using integer month as a linear feature**  
Month=12 (December) is numerically adjacent to month=1 (January) in reality but at opposite ends of a 1–12 scale. A linear model treats them as maximally different. Always encode cyclic variables (month, hour, day-of-week) as sine/cosine pairs.

**3. Computing ratio features that divide by zero**  
Ratio features like N:P often produce inf or NaN when the denominator is zero or near-zero. Always clip the denominator with `.clip(lower=min_val)` before dividing. NaN values from ratios will silently propagate through many sklearn transformers.

**4. Applying log transform to variables that contain zeros**  
`np.log(0)` returns `-inf`. Use `np.log1p(x)` (which computes log(1+x)) for variables that include zeros, or add a small constant before logging. Check `(X[col] <= 0).sum()` before applying any log transform.

**5. Engineering features after the train/test split using test-set statistics**  
Standardising, binning, or encoding using statistics computed on the full dataset (including the test set) introduces leakage. All feature engineering steps that involve computing statistics from data must be fitted on the training set only and then applied to the test set.
---
*python_methods_library - Samantha McGarrigle*